# 00 - Phase 1 Auto Run

Single-run Colab workflow: holdout split, experiment matrix training, diagnostics bundle, and auto-review files.

In [ ]:
import torch, sys
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime -> T4/A100"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
REPORT_DIR  = f'{DRIVE_BASE}/reports'
os.makedirs(f'{DATA_DIR}/processed', exist_ok=True)
os.makedirs(CHECKPT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

In [ ]:
import os, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}
sys.path.insert(0, REPO_DIR)
print('After git pull, use Runtime -> Restart session before trusting changed imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR
print('Policy review status:', cfg.get('experiments', {}).get('policy_review', {}).get('status'))

In [ ]:
AUTO_UNASSIGN = True
WRITE_FULL_BUNDLE = False  # Keep A100/Drive time low; set True for deep per-profile diagnostic zips.

import pandas as pd
from google.colab import runtime
from yenibot.experiments import (
    experiment_settings,
    prepare_training_holdout_split,
    run_experiment_matrix,
    write_experiment_diagnostics,
)

try:
    labeled_full = pd.read_parquet(f'{DATA_DIR}/processed/labeled_1h.parquet')
    holdout_path = f'{DATA_DIR}/processed/holdout_1h.parquet'
    labeled, holdout, holdout_meta = prepare_training_holdout_split(
        labeled_full,
        cfg,
        holdout_path=holdout_path,
    )
    cfg.setdefault('experiments', {})['holdout'] = holdout_meta
    min_selection_rows = (
        cfg['walk_forward']['train_bars']
        + cfg['walk_forward']['val_bars']
        + cfg['walk_forward']['test_bars']
        + cfg['walk_forward']['purge_bars']
        + cfg['walk_forward']['embargo_bars']
    )
    if len(labeled) <= min_selection_rows:
        raise ValueError(f'Not enough selection rows after holdout split: {len(labeled)}')

    settings = experiment_settings(cfg)
    print('Full labeled rows:', len(labeled_full))
    print('Selection/training rows:', len(labeled))
    print('Holdout rows:', len(holdout))
    print('Selection data end:', holdout_meta['selection_data_end'])
    print('Holdout data start:', holdout_meta['holdout_data_start'])
    print('Future OOS ready:', holdout_meta.get('future_oos_ready'))
    print('Future OOS next action:', holdout_meta.get('next_action'))
    print('Experiment mode:', settings.get('mode', 'staged'))
    print('Control profile:', settings.get('control_profile'))
    print('Candidate profiles:', settings.get('candidate_profiles', []))
    print('Always-full profiles:', settings.get('always_full_profiles', []))
    print('Seed audit:', settings.get('seed_audit', {}))

    train_result = run_experiment_matrix(
        labeled,
        cfg,
        checkpoint_dir=CHECKPT_DIR,
    )
    print('Experiment run:', train_result['run_id'])
    print('Experiment dir:', train_result['run_dir'])
    print('Training executed scopes:', train_result.get('training_executed_count'))
    print('Training reused scopes:', train_result.get('training_skipped_count'))

    diagnostics = write_experiment_diagnostics(
        checkpoint_dir=CHECKPT_DIR,
        config=cfg,
        output_dir=REPORT_DIR,
        run_id=train_result['run_id'],
        write_full_bundles=WRITE_FULL_BUNDLE,
    )
    display(diagnostics['comparison'])
    if not diagnostics.get('profile_blend').empty:
        display(diagnostics['profile_blend'])
    if not diagnostics.get('holdout_evaluation').empty:
        print('Holdout evaluation:')
        display(diagnostics['holdout_evaluation'])

    print('Full bundle enabled:', diagnostics.get('write_full_bundles'))
    print('Single download bundle:', diagnostics.get('bundle_zip'))
    print('Latest bundle shortcut:', diagnostics.get('latest_bundle_zip'))
    print('Slim summary bundle:', diagnostics.get('slim_bundle_zip'))
    print('Latest slim shortcut:', diagnostics.get('latest_slim_bundle_zip'))
    print('Auto review:', diagnostics.get('auto_review_path'))
    print('Next actions:', diagnostics.get('next_actions_path'))
    print('Decision:', diagnostics['decision'])
finally:
    if AUTO_UNASSIGN:
        runtime.unassign()
